# 04 - Difference-in-differences

Everything so far required measuring the confounders. Panel data offers an
escape: if we observe the same units before and after a policy, each group can
serve as its own control for anything that does not change over time — including
things we never measured.

What replaces conditional ignorability is **parallel trends**: the assumption
that, absent the policy, the two groups' outcomes would have moved together. It
is not weaker than ignorability so much as different, and it fails in ways that
are easy to check and easy to skip.

## Causal question

A policy is introduced for one group of units at a known date and not for
another. What was its effect on the outcome for the units that received it?

## Data and design

- **Unit of analysis:** one unit observed in each period — a balanced panel.
- **Group:** `treated_group`, fixed over time.
- **Time:** `time`, with `post` marking periods after the policy.
- **Treatment:** `treatment`, on only for treated units in post periods.
- **Outcome:** `outcome`.

The design has four cells — treated/control crossed with before/after — and the
estimate is the difference between the two changes.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_did_panel
from causal_inference_lab.difference_in_differences import difference_in_differences

dataset = make_did_panel(n_units=600, n_periods=8, seed=7)
data = dataset.data

print(f"units:        {data['unit'].nunique():,}")
print(f"periods:      {data['time'].nunique()}")
print(f"treated units:{data.loc[data['treated_group'] == 1, 'unit'].nunique():,}")
print(f"true effect:  {dataset.true_ate:.3f}")
print()
print("mean outcome by group and period:")
print(
    data.groupby(["time", "treated_group"])["outcome"]
    .mean()
    .unstack()
    .rename(columns={0: "control", 1: "treated"})
    .to_string(float_format=lambda v: f"{v:.2f}")
)

**Interpretation.** The treated group starts higher and stays higher, so a
cross-sectional comparison in any single period would confound the policy with
whatever made these units different to begin with. What matters for
difference-in-differences is not the levels but whether the two series move in
parallel before the policy — visible in the early periods, and tested below.

## Estimand

The **average treatment effect on the treated (ATT)**: the effect of the policy
on the units that actually received it.

Difference-in-differences cannot deliver an ATE. It says nothing about how
control units would have responded, because they never did.

## Identification assumptions

1. **Parallel trends.** Absent the policy, treated and control outcomes would
   have followed the same trajectory. Untestable after treatment; partially
   checkable before it.
2. **No anticipation.** Units did not change behaviour before the policy in
   expectation of it.
3. **Stable composition.** The same units are present throughout, and group
   membership does not change.
4. **No spillover.** The policy does not affect control units.

Assumption 1 is the whole design. Note carefully what a pre-trend test does and
does not establish: parallel movement before treatment is consistent with
parallel trends afterwards, but does not imply it.

## Estimation

The estimator computes the four cell means and takes the difference of
differences. We also compute the naive post-period comparison, to show what the
design is correcting.

In [ ]:
result = difference_in_differences(data)
effect = result.effect

post = data.loc[data["post"] == 1]
naive_post = (
    post.loc[post["treated_group"] == 1, "outcome"].mean()
    - post.loc[post["treated_group"] == 0, "outcome"].mean()
)

print(f"treated:  {result.pre_treated_mean:.3f} -> {result.post_treated_mean:.3f}  "
      f"(change {result.post_treated_mean - result.pre_treated_mean:+.3f})")
print(f"control:  {result.pre_control_mean:.3f} -> {result.post_control_mean:.3f}  "
      f"(change {result.post_control_mean - result.pre_control_mean:+.3f})")
print()
print(f"difference-in-differences: {effect.estimate:.3f}   (estimand {effect.estimand})")
print(f"true effect:               {dataset.true_ate:.3f}")
print(f"naive post-period gap:     {naive_post:.3f}")

**Interpretation.** The treated group rises by 5.04 and the control group by
2.06. The difference between those changes, 2.98, is the estimate, against a
true effect of 3.00.

The naive post-period comparison gives 3.22 — biased upward by 0.24, because it
attributes the pre-existing level gap between groups to the policy. That
pre-existing gap is precisely what differencing removes, and it is removed
without anyone having to measure why it existed.

## Diagnostics

The pre-trend test is the central diagnostic: it fits the two groups' trends in
the pre-treatment periods and asks whether they differ.

In [ ]:
print(f"pre-trend slope difference: {result.pre_trend_slope_difference:+.4f}")
print(f"p-value:                    {result.pre_trend_p_value:.3f}")

pre = data.loc[data["post"] == 0]
trend = (
    pre.groupby(["time", "treated_group"])["outcome"]
    .mean()
    .unstack()
    .rename(columns={0: "control", 1: "treated"})
)
trend["gap"] = trend["treated"] - trend["control"]
print()
print("pre-treatment gap between groups, by period:")
print(trend.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\ngap drift across pre-periods: "
      f"{trend['gap'].iloc[-1] - trend['gap'].iloc[0]:+.3f}")

**Interpretation.** The slope difference is −0.013 with a p-value of 0.78 — no
detectable divergence before the policy, and the period-by-period gap stays
close to constant.

Two cautions about reading this as reassurance. A high p-value is not evidence
of parallel trends; with few pre-periods the test has little power, so failing to
detect divergence is cheap. And parallel trends is a statement about the
*counterfactual* post-period, which no pre-period data can observe. The test can
refute the design; it cannot confirm it.

A placebo check strengthens the case: pretend the policy arrived earlier, during
a period when nothing actually happened. A well-behaved design should find no
effect.

In [ ]:
pre_only = data.loc[data["post"] == 0].copy()
periods = sorted(pre_only["time"].unique())
fake_switch = periods[len(periods) // 2]

pre_only["post"] = (pre_only["time"] >= fake_switch).astype(int)
pre_only["treatment"] = pre_only["post"] * pre_only["treated_group"]

placebo = difference_in_differences(pre_only)

print(f"placebo switch at period {fake_switch} (nothing happened here)")
print(f"placebo estimate: {placebo.effect.estimate:+.3f}")
print(f"real estimate:    {effect.estimate:+.3f}")
print(f"ratio:            {abs(placebo.effect.estimate / effect.estimate):.3f}")

**Interpretation.** The placebo estimate is essentially zero next to the real
one. Shifting the policy date into a period where nothing happened produces no
effect, which is what a valid design should do and what a design contaminated by
divergent trends would fail.

This is a genuinely informative check, because a spurious trend difference would
show up here as a non-zero placebo effect.

## Uncertainty

The estimator returns a point estimate. A block bootstrap that resamples whole
units — not individual rows — respects the panel structure, since observations
of the same unit across periods are not independent.

In [ ]:
units = data["unit"].unique()
rng = np.random.default_rng(7)

estimates = []
for _ in range(200):
    drawn = rng.choice(units, size=len(units), replace=True)
    resample = pd.concat(
        [data.loc[data["unit"] == unit].assign(unit=f"{unit}_{i}") for i, unit in enumerate(drawn)],
        ignore_index=True,
    )
    estimates.append(difference_in_differences(resample).effect.estimate)

estimates = np.array(estimates)
lower, upper = np.percentile(estimates, [2.5, 97.5])

print(f"estimate:      {effect.estimate:.3f}")
print(f"95% interval:  [{lower:.3f}, {upper:.3f}]")
print(f"standard error:{estimates.std():.3f}")
print(f"true effect:   {dataset.true_ate:.3f}")
print(f"covers truth:  {lower <= dataset.true_ate <= upper}")

**Interpretation.** The interval covers the true effect. Resampling units rather
than rows is what makes it trustworthy: rows within a unit are correlated, so
naive row-level resampling would understate the standard error and produce an
interval that is too narrow.

The interval reflects sampling variability only. If parallel trends fails, the
estimate is biased and this interval will sit confidently around the wrong
number.

## Limitations

- **Parallel trends is assumed, not established.** The pre-trend test and
  placebo both pass, which is necessary and not sufficient. Neither observes the
  counterfactual that matters.
- **The pre-trend test has limited power.** With four pre-periods, moderate
  divergence would go undetected. A high p-value is weak evidence.
- **Two groups, one switch date.** Staggered adoption, where units are treated
  at different times, requires estimators not implemented here — the two-way
  fixed effects estimate can be badly biased in that setting.
- **No covariates.** This is unconditional DiD. Conditional parallel trends,
  where the assumption holds only after adjustment, needs a different estimator.
- **Spillover would be invisible.** If the policy affected control units, their
  trend shifts too and the estimate absorbs the difference.
- **The estimand is the ATT.** It says nothing about what the policy would do if
  extended to the control units.